In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from itertools import product

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '4_Baselines' / '4.3_OutcomeModel'))
import MF_class as MF
import OM_class as OM

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### 0. Choose Dataset

In [2]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Simulation'

# 1. Load Data

In [ ]:
# Load train and test data for MF
train = pd.read_csv(data_path / 'train.csv')
test = pd.read_csv(data_path / 'test.csv')

n_users = train['user_id'].nunique()
n_items = train['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

test_users = test['user_id'].unique()
all_users = train['user_id'].unique()
train_users = np.setdiff1d(all_users, test_users)

gt = pd.read_csv(data_path / 'ground_truth_processed.csv')
oracle_dict = gt.set_index(['cause_id', 'effect_id']).to_dict()['causal_effect']

Number of users: 6040, Number of items: 3952


In [6]:
test_items = test['item_id'].unique()
test_pairs = list(product(test_items, test_items))

good_pairs = []
for pair in test_pairs:
    if pair in oracle_dict:
        good_pairs.append(pair)
print(f"Total item pairs in oracle: {len(good_pairs)}")

bad_pairs = list(set(test_pairs) - set(good_pairs))
print(f"Total item pairs not in oracle: {len(bad_pairs):,}")

rng = np.random.default_rng(seed=42)
chosen_null_pairs = rng.choice(bad_pairs, size=10000, replace=False)
chosen_null_pairs = [tuple(pair) for pair in chosen_null_pairs]

all_pairs = good_pairs + chosen_null_pairs
print(f"Total item pairs to evaluate: {len(all_pairs):,}")

Total item pairs in oracle: 613
Total item pairs not in oracle: 623,487
Total item pairs to evaluate: 10,613


# 2. Create Training and Validation Sets for Outcome Model

In [12]:
def get_timestamp_dict(data, users):

    subset = data[data['user_id'].isin(users)]
    pos = (
        subset[subset['watched'] == 1]
        .groupby(['user_id', 'item_id'])['timestamp']
        .first()
    )

    timestamp_dict = {user_id : user_pos.droplevel(0).to_dict() for user_id, user_pos in tqdm(pos.groupby(level=0))}
    for user_id in users:
        if user_id not in timestamp_dict:
            timestamp_dict[user_id] = {}

    return timestamp_dict

def get_om_data(data, users, item_pairs_id, users_per_pair, random_state=42): 
    rng = np.random.default_rng(seed=random_state)
    len_users = len(users) 
    
    print("Constructing timestamp dictionary...")
    timestamp_dict = get_timestamp_dict(data, users)
    
    print("Generating OM training data...")
    rows = []
    for (i, j) in tqdm(item_pairs_id):
        for _ in range(users_per_pair):
            user = users[rng.integers(len_users)]
            pos_items_dict = timestamp_dict[user]

            x, y = 0, 0
            if i in pos_items_dict:
                if j in pos_items_dict:
                    if pos_items_dict[i] < pos_items_dict[j]:
                        x, y = 1, 1
                    else:
                        continue
                else:
                    x = 1
            elif j in pos_items_dict:
                y = 1
                    
            rows.append((user, i, j, x, y)) 
    
    return pd.DataFrame(rows, columns=["u", "i", "j", "x", "y"])

In [13]:
users_per_pair = 300

om_train_data = get_om_data(
    data=train, 
    users=train_users, 
    item_pairs_id=all_pairs, 
    users_per_pair=users_per_pair, 
    random_state=42)

om_test_data  = get_om_data(
    data=test,  
    users=test_users, 
    item_pairs_id=all_pairs, 
    users_per_pair=users_per_pair, 
    random_state=42)

Constructing timestamp dictionary...


  0%|          | 0/3020 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/10613 [00:00<?, ?it/s]

Constructing timestamp dictionary...


  0%|          | 0/3020 [00:00<?, ?it/s]

Generating OM training data...


  0%|          | 0/10613 [00:00<?, ?it/s]

# 3. Save Datasets

In [15]:
om_train_data.to_csv(data_path / 'om_train.csv', index=False)
om_test_data.to_csv(data_path / 'om_test.csv', index=False)